<div align="left">
    <img src="https://form.ethz.ch/hss26/_jcr_content/par/image/image.imageformat.1286.dpr2.1527502072.png" alt="Earth Observation Foundation Models Workspace" width="70%" style="max-height: 350px; object-fit: cover; border-radius: 8px; box-shadow: 0 4px 10px rgba(0,0,0,0.15);">
</div>

# Hands-on Session: Earth Observation (EO) Foundation Models

🌍 **Can an AI model read the Earth's surface — without ever being told what to look for?**
In this hands-on session, we explore geospatial foundation models and put them to work on real satellite imagery — extracting deep structural and spectral fingerprints of the landscape across time, reducing millions of features into interpretable patterns, and letting the model reveal what changed, when, and where. No labels. No manual rules. **Just intelligence learned from the Earth itself.**

### 🔍 Technical Workflow: Framework Execution

**Case study:** Trentino region — Val di Fiemme forest, devastated by the **Vaia storm (October 2018)**.  
We use this event as a ground-truth anchor to validate whether the pipeline can autonomously detect the abrupt landscape change without any labelled data.

**Pipeline steps:**

1. **Load & process HLS data from GEE**
   - Query the Harmonized Landsat Sentinel-2 (HLS) collection from Google Earth Engine
   - Filter by area of interest (Val di Fiemme) and time range spanning pre- and post-Vaia (2017–2019)

2. **Load the foundation model**
   - Initialize pre-trained weights of **Prithvi EO v2** via the `terratorch` registry (`prithvi_eo_v2_300`)

3. **Generate high-dimensional embeddings**
   - Pass image tensors through the Vision Transformer (ViT) encoder
   - Each observation date is mapped to a dense feature vector — a mathematical "fingerprint" of the landscape at that moment

4. **Feature reduction & visualization (PCA & t-SNE)**
   - **PCA** isolates the directions of maximum variance across dates, surfacing the most structurally significant changes
   - **t-SNE** projects the high-dimensional space into 2D, making temporal clusters and outliers (e.g. the post-Vaia observations) visually separable

5. **Change date detection**
   - Compare feature spaces across time steps
   - Identify the date where the embedding distribution shifts most abruptly — expected to align with **October 2018**

---

### 💡 Core Takeaways

#### 1. HLS data
- A radiometrically harmonized, analysis-ready product fusing Landsat and Sentinel-2
- Provides consistent multi-spectral time series — essential for temporal change analysis

#### 2. Geospatial foundation models
- **Multi-spectral design:** ingest any band combination (RGB, NIR, SWIR) in any order — not limited to 3-channel RGB
- **Self-supervised pre-training:** trained on millions of unlabelled global images via Masked Autoencoders (MAE), learning Earth surface physics with no manual annotation
- **Zero-shot capability:** no fine-tuning needed for downstream tasks like change detection

#### 3. Embeddings as landscape fingerprints
- Instead of pixel-by-pixel band comparisons, the model encodes the full structural pattern, canopy texture, and moisture status into a single vector
- Changes in the embedding space directly reflect real-world landscape changes — in our case, the mass windthrow caused by Vaia across Val di Fiemme.


## Requirements
- **Google Earth Engine** — free account + a GEE Cloud Project ID
- **Google Drive** — for exporting image chips (~100 MB)
- **Hugging Face** — free account + accept Prithvi EO v2 model license
- **GPU runtime** — Google Colab (T4) or local CUDA GPU (≥8 GB VRAM)




## Setup
Install required packages

In [2]:
# Install packages
!pip install geemap
!pip install --upgrade -q pip
!pip install -q opencv-python terratorch>=1.1.0 einops scikit-learn>=1.5.0

In [ ]:
#Load packages
import ee
import geemap

import cv2
import torch
import numpy as np
import time
from einops import rearrange, reduce
from matplotlib import pyplot as plt
from sklearn import decomposition, preprocessing
from terratorch import BACKBONE_REGISTRY
from sklearn.manifold import TSNE


### Step 1: Initialize GEE
We need `geemap` for interactive map visualizations natively within the Colab environment, and the `ee` library to communicate with the Google Earth Engine API.

### Prerequisites
Make sure you have access to a **Google Earth Engine (GEE)** account. When running the initialization cell, you will be prompted to authenticate through your browser.

### Google Earth Engine requires a Google Cloud Project ID to run.
#
### How to find your Project ID:
1. Open a new tab and go to: https://code.earthengine.google.com/
2. Look at the top-right corner next to your profile picture/name.
3. Copy the project text ID shown there (e.g., 'ee-yourusername' or 'my-project-1234').
4. Paste that text string inside the single quotes for PROJECT_ID below.

In [ ]:
# 1. Authenticate to Earth Engine
ee.Authenticate()

# 2. Define your Earth Engine Cloud Project ID here
# Students can find this on their Earth Engine code editor page or GCP console
PROJECT_ID = 'PROJECT_ID'

# 3. Initialize with the project ID
ee.Initialize(project=PROJECT_ID)

### Step 2: Define Region of Interest (ROI)
We define a point using a specific latitude and longitude, create a buffer of 3,840 meters around it, and find its bounding box coordinates (`bounds()`) to use as our target window.

In [ ]:
lat = 46.231128
lon = 11.407454

ROI = ee.Geometry.Point([lon, lat]).buffer(3840).bounds()

print("ROI defined successfully.")

### Step 3: Load and Filter Base Image Collection
We target the **NASA/HLS/HLSS30/v002** image collection (Harmonized Landsat Sentinel-2). To ensure we don't handle empty images or data fragments, we write a mapping function to count valid pixels inside our ROI using `reduceRegion` and filter out any completely blank scenes.

In [ ]:
# Load the raw collection overlapping our ROI
collection = ee.ImageCollection('NASA/HLS/HLSS30/v002').filterBounds(ROI)

# Define helper mapping function to get valid pixel counts inside our ROI boundary
def count_pixels(img):
    count = img.reduceRegion(
        reducer=ee.Reducer.count(),
        geometry=ROI,
        scale=30,
        maxPixels=1e13
    ).values().get(0)
    return img.set('pixel_count', count)

# Map the function and filter collection
validCollection = collection.map(count_pixels).filter(ee.Filter.gt('pixel_count', 0))

print("Collection filtered for valid pixels successfully.")

### Step 4: Image Selection Strategy (2018 vs 2019)
We select imagery from two fixed yearly windows: **2018** (pre-storm) and **2019** (post-storm). We filter out heavily cloudy scenes (keeping less than 30% cloud cover), sort by clarity, and select the **20 images from each year**, giving a final collection of 40 images.

In [ ]:
# Filter 20 clearest images from 2018 (pre-Vaia)
beforePool = (validCollection
              .filterDate('2018-01-01', '2018-12-31')
              .filter(ee.Filter.lt('CLOUD_COVERAGE', 30))
              .sort('CLOUD_COVERAGE'))

# Filter 20 clearest images from 2019 (post-Vaia)
afterPool = (validCollection
             .filterDate('2019-01-01', '2019-12-31')
             .filter(ee.Filter.lt('CLOUD_COVERAGE', 30))
             .sort('CLOUD_COVERAGE'))

# Select top 20 clearest from each year and merge
before = beforePool.limit(20)
after = afterPool.limit(20)
selected = before.merge(after)

print(f"2018 images:  {before.size().getInfo()}")
print(f"2019 images: {after.size().getInfo()}")
print(f"Total combined images:   {selected.size().getInfo()}")

### Step 5: Extracting Metadata & Dates
To make working with the images easier, we'll extract the system timestamp and parse it into a clean readable string format (`YYYY-MM-dd`). Then we will pull this date array into python's client side so you can easily choose which one to preview.

In [ ]:
# Map function to add readable text dates as metadata attribute
def add_date_str(img):
    date = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
    return img.set('date_str', date)

withDate = selected.map(add_date_str)

# Convert Server-side Collection to Server List and fetch array attributes to client
list_images = withDate.toList(40)
dates = withDate.aggregate_array('date_str').getInfo()

print("Available Scene Dates mapped to indices:")
for idx, date_str in enumerate(dates):
    print(f"Index [{idx}]: {date_str}")

### Step 6: Visual Inspection & Scene Selection

Before passing images into the model, we visually inspect each scene to hand-pick the **6 cleanest images per year** — free of snow and cloud cover.

Change `viewIndex` (0–39) to browse through all 40 images:
- **0–19** → 2018 scenes
- **20–39** → 2019 scenes

Scenes to **exclude**:
1. Heavy cloud cover obscuring the forest canopy
2. Snow cover altering the spectral signature

Note the indices of the 6 best scenes per year before proceeding to the next step.

In [ ]:
# CHANGE THIS INDEX VALUE TO TEST DIFFERENT TIMESTAMPS
viewIndex = 39

# Fetch the specified target image
imageToVisualize = ee.Image(list_images.get(viewIndex)).clip(ROI)
imageDateStr = dates[viewIndex]

# Build interactive map workspace
Map = geemap.Map()
Map.centerObject(ROI, 12)

# Configure True Color RGB Render parameter values
vis_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0.005,
    'max': 0.2,
    'gamma': 1.9
}

Map.addLayer(imageToVisualize, vis_params, f'HLS RGB Scene [{viewIndex}]: {imageDateStr}')
print(f"Displaying Scene captured on: {imageDateStr}")
Map

### STEP 6b:  SELECTION (6 images from 2018 & 6 images from 2019)

### INSTRUCTIONS:

Based on your visual inspection using the map widget above, choose:


1.   Less cloudy 6 image indices from 2018 (Indices 0 to 19)
2.   Less cloudy 6 image indices from 2019 (Indices 20 to 39).

Replace the numbers in the lists below with your chosen index choices.

In [ ]:
# Example selection: Enter your chosen indices here
chosen_2018_indices = [0, 2, 6, 7, 10, 11]     # Must be indices between 0 and 19
chosen_2019_indices  = [22, 23, 24, 27, 33, 38] # Must be indices between 20 and 39

# Combine the choices into a single index list
final_indices = chosen_2018_indices + chosen_2019_indices

# Basic validation check for the students
if len(chosen_2018_indices) != 6 or len(chosen_2019_indices) != 6:
    print("[WARNING]: Please ensure you select exactly 6 images for both pools!")

# Extract the chosen images from our master list using Earth Engine server logic
final_image_list = []
final_dates_list = []

print("Extracting your final curated dataset collection...")
for index in final_indices:
    final_image_list.append(ee.Image(list_images.get(index)))
    final_dates_list.append(dates[index])
    print(f"-> Retained Index [{index}]: Captured on {dates[index]}")

# Convert our selected list references back into a workable Server List object
# so the down-stream export code works perfectly
list_images = ee.List(final_image_list)
dates = final_dates_list

print("\n[SUCCESS]: Final 12 scenes locked in. Proceed to the batch export cell below.")

### Step 7: Batch Export to Google Drive

Before running the foundation model locally, we export the selected scenes from GEE to Google Drive as analysis-ready image chips.

**What gets exported:**
- **Bands:** `B2, B3, B4, B8A, B11, B12` — the exact 6-band combination expected by Clay (Blue, Green, Red, Narrow NIR, SWIR-1, SWIR-2)
- **Size:** 256×256 pixels per chip — the native patch size of the ViT encoder
- **Projection:** EPSG:32632 (UTM Zone 32N — appropriate for the Val di Fiemme area)
- **Format:** Float32, one file per date, named `HLS_Clay_YYYY-MM-DD`

One export task is submitted per image (12 total). Tasks run asynchronously in the GEE backend — monitor progress in the **GEE Task Manager** before moving to the next step.

In [ ]:
for i in range(12):
    img = ee.Image(list_images.get(i))
    dateStr = dates[i]

    # Filter down specific channel combinations for Machine Learning
    mlInput = img.select(['B2', 'B3', 'B4', 'B8A', 'B11', 'B12']).toFloat()

    # Setup Drive Export Pipeline profile parameter options
    task = ee.batch.Export.image.toDrive(
        image=mlInput.clip(ROI),
        description=f'HLS_Clay_{dateStr}',
        fileNamePrefix=f'HLS_Clay_{dateStr}',
        region=ROI,
        dimensions=(256, 256),
        crs='EPSG:32632',
        maxPixels=1e13
    )

    # Fire processing request parameters straight into asynchronous Engine worker queues
    task.start()
    print(f"Submitted batch processing pipeline job: HLS_Clay_{dateStr}")

print("\nAll 12 background engine threads pushed to tracking workspace successfully! Open your GEE dashboard or task runner to review file rendering updates.")

# Section 2: Generating Deep Geospatial Embeddings with Prithvi

Now that we have selected and verified our curated sequence of clean HLS images, we will leverage **Prithvi**—a state-of-the-art geospatial foundation model built by NASA and IBM.

Instead of treating pixels as isolated spectral numbers, Prithvi uses a 3D Vision Transformer (ViT) architecture to look at patches of your imagery across space and time simultaneously. It outputs dense multi-dimensional vector representations called **embeddings** that summarize semantic surface features (e.g., changes in vegetation, soil moisture, and land surface features caused by the storm).

### Step 8: Initialize Modeling Runtime Environment
We check for hardware device acceleration (CUDA GPU) and ensure the necessary geospatial model modules are ready for execution.

We installed the `terratorch` library framework along with `einops` for tensor reshaping operations, and `scikit-learn` to handle downstream evaluation tasks.

In [ ]:
# Verify if GPU acceleration is active
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using hardware device acceleration: {device}")
print(f"Active CV2 Version: {cv2.__version__}")

### Step 9: Instantiate and Load Pretrained Prithvi Model Weights
We pull the target backbone model (`prithvi_eo_v2_300`) directly from the registry, set it to evaluation mode, and move it to our active runtime hardware processing device.

In [ ]:
# Verify that our targeted backbone exists in the terratorch framework registry
model_name_target = "prithvi_eo_v2_300"
assert any(model_name_target in m for m in BACKBONE_REGISTRY), "Target model backbone not found!"

print("--> Downloading and instantiating Prithvi model weights from Hugging Face (~1.3 GB)...")
# Build the model using the registry with pretrained weights
model = BACKBONE_REGISTRY.build(model_name_target, pretrained=True)
model.to(device)
model.eval() # Set model to evaluation mode

print(f"\n[SUCCESS]: {model_name_target} successfully loaded and allocated to memory.")

### Step 10: Format & Shape Selected Imagery to Sequence Chips

Prithvi EO v2 expects inputs structured as tensor sequences: `[Batch, Channels, Time, Height, Width]`.

We prepare the data in three stages:

1. **Sort & download** — images are sorted chronologically and downloaded from GEE at native 30 m resolution, then resized band-by-band to exactly **256×256** using bilinear interpolation

2. **Stack** — all 12 scenes are stacked into a single array of shape **(12, 6, 256, 256)** — 12 dates × 6 bands × 256 × 256 pixels

3. **Sliding window chipping** — rather than fixed non-overlapping blocks, a window of **4 timesteps** slides one step at a time across the 12 scenes, producing **9 overlapping chips**. Each chip captures a short temporal sequence and consecutive chips share 3 images, giving the model finer temporal resolution to detect gradual or abrupt changes:

| Chip | Indices | Description |
|------|---------|-------------|
| 0 | 0 → 3 | Early 2018 |
| 1 | 1 → 4 | Mid 2018 |
| ... | ... | ... |
| 5 | 5 → 8 | Transition (pre → post Vaia) |
| ... | ... | ... |
| 8 | 8 → 11 | 2019 post-storm |

The chip covering the **October 2018** boundary is where we expect the largest shift in the embedding space.

In [ ]:
print("1. Enforcing chronological order on selected images...")
# Sort server-side by metadata timestamp to ensure true time-series order
temp_collection = ee.ImageCollection(list_images).sort('system:time_start')

sorted_list_images = temp_collection.toList(12)
sorted_dates = temp_collection.aggregate_array('date_str').getInfo()

print("\n2. Converting Earth Engine sorted tensors into localized numpy sequences...")

image_arrays = []
for i in range(12):
    img_ee = ee.Image(sorted_list_images.get(i))
    mlInput = img_ee.select(['B2', 'B3', 'B4', 'B8A', 'B11', 'B12'])

    # Download raw pixels
    img_np = geemap.ee_to_numpy(mlInput, region=ROI, scale=30)

    # LOCAL RESIZING WORKAROUND: Force array dimensions to exactly 256x256 band-by-band.
    resized_bands = []
    for c in range(img_np.shape[2]):
        band_resized = cv2.resize(img_np[:, :, c], (256, 256), interpolation=cv2.INTER_LINEAR)
        resized_bands.append(band_resized)
    img_np = np.stack(resized_bands, axis=-1)

    # The output array shape is now exactly (256, 256, 6) -> transpose to (6, 256, 256)
    img_np = np.transpose(img_np, (2, 0, 1))
    image_arrays.append(img_np)

# Stack all images along a new time dimension (Shape: 12, 6, 256, 256)
full_stack = np.stack(image_arrays, axis=0)

# ==============================================================================
# SLIDING WINDOW LOGIC: 4 timesteps per window, advancing 1 timestep at a time
# ==============================================================================
window_size = 4
chips = []

# Loop from index 0 to 8 to create 9 overlapping chips
for start_idx in range(len(full_stack) - window_size + 1):
    end_idx = start_idx + window_size
    chip_slice = full_stack[start_idx:end_idx]  # Slices a (4, 6, 256, 256) tensor
    chips.append(chip_slice)

print(f"\n[SUCCESS]: Generated {len(chips)} overlapping sliding window chips.")
print(f"Shape of individual chip sequence tensor (Time, Bands, Height, Width): {chips[0].shape}\n")

# ==============================================================================
# PRINT ROLLING CHIP DATE BREAKDOWN FOR PRITHVI TENSORS
# ==============================================================================
print("====== PRITHVI TENSOR SEQUENCE SLIDING BREAKDOWN ======")
for i, start_idx in enumerate(range(len(chips))):
    end_idx = start_idx + window_size
    window_dates = sorted_dates[start_idx:end_idx]

    print(f"📦 CHIP {i} (Indices {start_idx} to {end_idx-1}) Tensors formed by:")
    print(f"  └─► {', '.join(window_dates)}")
print("=======================================================")

### Step 11: Pass Chips Through Prithvi to Extract Embeddings
We execute the forward processing pipeline on the GPU workspace, extract the generalized spatial representation `[CLS]` token for each timeline sequence block, and construct our multi-dimensional embeddings map matrix.

In [ ]:
embeddings_list = []
start_time = time.time()

print("Executing forward pass inference to extract global latent representations...")

for idx, chip in enumerate(chips):
    # Format dimensions from (T, C, H, W) to (Batch=1, C, T, H, W) to match Prithvi framework
    chip_tensor = rearrange(chip, "t c h w -> 1 c t h w")
    chip_tensor = torch.from_numpy(chip_tensor).float().to(device)

    with torch.no_grad(): # Disable gradient collection to conserve memory
        # Forward pass output representation token maps
        output_embedding = model.forward(chip_tensor)

        # Extract the Class Token [CLS] at position index 0 representing the global summarized view of the chip
        cls_token = output_embedding[0][:, 0, :].detach().cpu().numpy().ravel()
        embeddings_list.append(cls_token)
        print(f"-> Generated embedding vector for Chip [{idx}] with dimensions: {cls_token.shape}")

# Convert list into a structured multi-dimensional matrix map
embeddings_matrix = np.array(embeddings_list)
print(f"\n[SUCCESS]: Processed all matrices in {time.time() - start_time:.2f} seconds.")

### Step 12: Embedding Extraction & PCA Trajectory Analysis

With the 9 sliding window chips ready, we pass each through Prithvi EO v2 and analyse how the embedding space evolves over time.

**1. Embedding extraction**
Each chip of shape `(1, 6, 4, 256, 256)` is fed into the model. The output feature tensor is flattened into a single vector per window, producing a **(9, N)** embedding matrix — one row per temporal window.

**2. Normalization & PCA**
The matrix is standardized with `StandardScaler` and projected onto the **top 2 principal components**:
- **PC1** captures the dominant axis of landscape variance — typically driven by large structural changes such as canopy loss
- **PC2** captures secondary variation — often seasonal effects, moisture, or partial disturbance

**3. Trajectory visualization**
The two panels plot each window's PCA coordinate over time (W0 → W8), colored blue-to-red chronologically:
- A **smooth trajectory** indicates gradual seasonal change
- An **abrupt jump** between consecutive windows signals a sudden landscape shift — expected around the window spanning **October 2018**

> 💡 *Look for the window where PC1 or PC2 shows the largest single step — that window contains the Vaia storm transition.*

In [ ]:
print("1. Extracting high-dimensional embeddings from the 9 true sliding window chips...")
rolling_embeddings = []

# chips contains 9 segments of shape (4, 6, 256, 256) from your working Step 10
for i in range(len(chips)):
    chip_sequence = chips[i]  # Shape: (Time=4, Bands=6, H=256, W=256)

    # Rearrange layout to match Prithvi: [Channels, Time, Height, Width]
    chip_transposed = np.transpose(chip_sequence, (1, 0, 2, 3)) # Becomes (6, 4, 256, 256)

    # Add batch dimension -> Shape becomes (1, 6, 4, 256, 256)
    input_tensor = torch.from_numpy(chip_transposed).float().unsqueeze(0).to(device)

    with torch.no_grad():
        try:
            features = model(input_tensor)
        except NameError:
            features = backbone(input_tensor)

        # Unpack output layers safely
        if isinstance(features, list):
            feature_tensor = features[-1]
        elif isinstance(features, dict):
            first_key = list(features.keys())[0]
            feature_tensor = features[first_key]
        else:
            feature_tensor = features

        # Extract to CPU and flatten completely
        feature_np = feature_tensor.detach().cpu().numpy()
        flat_vector = feature_np.flatten()
        rolling_embeddings.append(flat_vector)

# Create the 2D matrix safely in local memory
rolling_matrix = np.vstack(rolling_embeddings)
print(f"[SUCCESS]: Generated matrix of shape: {rolling_matrix.shape}")

# ==============================================================================
# 2. STANDARD SCALING & MULTI-COMPONENT PCA
# ==============================================================================
print("\n2. Normalizing feature variances...")
scaler = preprocessing.StandardScaler()
scaled_matrix = scaler.fit_transform(rolling_matrix)

print("3. Extracting top 2 principal components to decouple environmental factors...")
pca_rolling = decomposition.PCA(n_components=2)
pca_results = pca_rolling.fit_transform(scaled_matrix)

var1 = pca_rolling.explained_variance_ratio_[0] * 100
var2 = pca_rolling.explained_variance_ratio_[1] * 100
print(f"   └─► Component 1 accounts for: {var1:.1f}% variance")
print(f"   └─► Component 2 accounts for: {var2:.1f}% variance")

# ==============================================================================
# 3. SIDE-BY-SIDE TRAJECTORY VISUALIZATION
# ==============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6))
window_colors = plt.cm.coolwarm(np.linspace(0, 1, len(chips)))
window_ticks = [f"W{i}" for i in range(len(chips))]

# --- Plot PCA Component 1 ---
for i in range(len(chips)):
    ax1.scatter(i, pca_results[i, 0], color=window_colors[i], s=200, edgecolors='black', zorder=3)
ax1.plot(range(len(chips)), pca_results[:, 0], color='black', linestyle='--', alpha=0.5, zorder=2)
ax1.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax1.set_xticks(range(len(chips)), window_ticks)
ax1.set_xlabel("Sliding Window Chronological Steps")
ax1.set_ylabel("PCA Component 1 Coordinate")
ax1.set_title(f"PCA Component 1 Variance Path ({var1:.1f}%)")
ax1.grid(True, alpha=0.2)

# --- Plot PCA Component 2 ---
for i in range(len(chips)):
    label_text = f"W{i}: {sorted_dates[i]} to {sorted_dates[i+3]}"
    ax2.scatter(i, pca_results[i, 1], color=window_colors[i], s=200, edgecolors='black', label=label_text, zorder=3)
ax2.plot(range(len(chips)), pca_results[:, 1], color='black', linestyle='--', alpha=0.5, zorder=2)
ax2.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax2.set_xticks(range(len(chips)), window_ticks)
ax2.set_xlabel("Sliding Window Chronological Steps")
ax2.set_ylabel("PCA Component 2 Coordinate")
ax2.set_title(f"PCA Component 2 Variance Path ({var2:.1f}%)")
ax2.grid(True, alpha=0.2)

# Place legend cleanly outside the subplots
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Window Span Dates")
plt.tight_layout()
plt.show()


### Step 13: Per-Scene Embedding Extraction & Dimensionality Reduction

We extract one embedding per scene (12 total). Since Prithvi requires **4 timesteps** as input, each single scene is replicated 4× along the time dimension as a workaround — the model still encodes the spectral and structural fingerprint of the scene, but without true temporal context.

The resulting **(12, N)** matrix is then reduced using PCA and t-SNE:
- **PC1** — tracks atmospheric/illumination variance across dates
- **PC2** — tracks canopy and structural landscape state
- **t-SNE** — groups scenes by semantic similarity; pre- and post-Vaia scenes should form two distinct clusters


In [ ]:
print("1. Extracting high-dimensional structural representations from 'full_stack'...")
individual_embeddings = []

for i in range(12):
    single_image = full_stack[i] # Native Shape: (6, 256, 256)

    # Structure tensor to match Prithvi format: [Batch, Channels, Time, Height, Width]
    single_date_sequence = np.expand_dims(single_image, axis=1)
    single_date_sequence = np.repeat(single_date_sequence, 4, axis=1) # 4x Time replication workaround
    input_tensor = torch.from_numpy(single_date_sequence).float().unsqueeze(0).to(device)

    with torch.no_grad():
        try:
            features = model(input_tensor)
        except NameError:
            features = backbone(input_tensor)

        # Unpack layers safely based on your model's output configuration wrapper
        if isinstance(features, list):
            feature_tensor = features[-1]
        elif isinstance(features, dict):
            first_key = list(features.keys())[0]
            feature_tensor = features[first_key]
        else:
            feature_tensor = features

        # Extract to CPU and flatten completely to preserve all spatial structural detail
        feature_np = feature_tensor.detach().cpu().numpy()
        flat_vector = feature_np.flatten()

        individual_embeddings.append(flat_vector.tolist())

# Force stacking into a true 2D matrix layout: shape (12, 1049600)
individual_embeddings_matrix = np.vstack([np.array(x) for x in individual_embeddings])
print(f"[SUCCESS]: Generated a true 2D matrix shape: {individual_embeddings_matrix.shape}")

# ==============================================================================
# 2. STANDARD SCALING
# ==============================================================================
print("\n2. Normalizing individual date feature variances...")
scaler = preprocessing.StandardScaler()
scaled_matrix = scaler.fit_transform(individual_embeddings_matrix)

# ==============================================================================
# 3. LINEAR DIMENSIONALITY REDUCTION (PCA)
# ==============================================================================
print("3. Projecting linear variance along orthogonal components...")
pca_individual = decomposition.PCA(n_components=2)
pca_ind_results = pca_individual.fit_transform(scaled_matrix)

var1 = pca_individual.explained_variance_ratio_[0] * 100
var2 = pca_individual.explained_variance_ratio_[1] * 100

# ==============================================================================
# 4. NON-LINEAR DIMENSIONALITY REDUCTION (t-SNE)
# ==============================================================================
print("4. Computing non-linear proximity manifolds via t-SNE...")
# Adaptively set perplexity safely below sample count constraints
safe_perplexity = min(3, len(individual_embeddings_matrix) - 1)

tsne = TSNE(n_components=2, random_state=42, perplexity=safe_perplexity, n_iter=1000)
tsne_results = tsne.fit_transform(scaled_matrix)
print("[SUCCESS]: Dimensionality reduction layers completed.")

# ==============================================================================
# 5. INTEGRATED VISUALIZATION PIPELINE (PCA PANELS + t-SNE SEPARATION)
# ==============================================================================
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(26, 7))
timeline_colors = plt.cm.coolwarm(np.linspace(0, 1, 12))

# --- Panel 1: PCA Component 1 (Atmospheric Track) ---
for i in range(12):
    ax1.scatter(i, pca_ind_results[i, 0], color=timeline_colors[i], s=250, edgecolors='black', zorder=3)
ax1.plot(range(12), pca_ind_results[:, 0], color='black', linestyle='-', alpha=0.4, linewidth=2, zorder=2)
ax1.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax1.set_xticks(range(12))
ax1.set_xticklabels(sorted_dates, rotation=45, ha='right')
ax1.set_xlabel("Exact Satellite Capture Date")
ax1.set_ylabel("PCA Component 1 Coordinate")
ax1.set_title(f"PCA Component 1 (Atmosphere: {var1:.1f}%)")
ax1.grid(True, alpha=0.2)

# --- Panel 2: PCA Component 2 (Canopy/Landscape Track) ---
for i in range(12):
    ax2.scatter(i, pca_ind_results[i, 1], color=timeline_colors[i], s=250, edgecolors='black', zorder=3)
ax2.plot(range(12), pca_ind_results[:, 1], color='black', linestyle='-', alpha=0.4, linewidth=2, zorder=2)
ax2.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax2.set_xticks(range(12))
ax2.set_xticklabels(sorted_dates, rotation=45, ha='right')
ax2.set_xlabel("Exact Satellite Capture Date")
ax2.set_ylabel("PCA Component 2 Coordinate")
ax2.set_title(f"PCA Component 2 (Canopy State: {var2:.1f}%)")
ax2.grid(True, alpha=0.2)

# --- Panel 3: t-SNE Non-Linear Neighborhood Groupings ---
for i in range(12):
    label_text = f"Day {i}: {sorted_dates[i]}"
    ax3.scatter(
        tsne_results[i, 0],
        tsne_results[i, 1],
        color=timeline_colors[i],
        s=300,
        edgecolors='black',
        label=label_text,
        zorder=3
    )
ax3.set_xlabel("t-SNE Dimension 1")
ax3.set_ylabel("t-SNE Dimension 2")
ax3.set_title("t-SNE Manifold Clusters (Semantic Proximity)")
ax3.grid(True, alpha=0.2)

# Position chronological legends cleanly outside the figure space
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Exact Capture Timeline")
plt.tight_layout()
plt.show()

print("Execution complete! Analyze Panel 3 to check if the pre-storm days and cleared-canopy days group into distinct cluster islands.")

#Exercises

### Exercise 1: Random Forest Classification on Embeddings

Instead of just visualising the feature space, can we **classify** each scene as pre- or post-Vaia using the embeddings as input features?

**How to do it:**
- Use `full_stack` (shape `12 × 6 × 256 × 256`) — flatten each scene's embedding into a 1D feature vector
- Assign binary labels: `0` for 2018 scenes, `1` for 2019 scenes
- Train a `RandomForestClassifier` from scikit-learn using a leave-one-out or train/test split

**What to expect:**
- High accuracy (ideally 100%) would confirm the embeddings carry enough discriminative information to separate pre- from post-storm scenes
- Low accuracy suggests the selected scenes are too noisy or too few — a good reason to go back and refine your scene selection

> 💡 *Hint: try adding the noisy scenes from Exercise 1 to your training set — does more data help or hurt the classifier?*

### Exercise 2: What Happens With Noisy Scenes?

Re-run the pipeline including **cloudy or snowy images** you excluded in Step 5 and observe how the embedding space reacts.

**How to do it:**
- Manually add back the snowy/cloudy scene indices you noted during visual inspection
- Re-run later steps with the new image set

**What to expect:**
- Cloudy scenes will likely appear as **outliers** in both PCA and t-SNE — cloud cover dominates the spectral signal and pushes the embedding far from clear-sky observations
- Snowy scenes will form their **own cluster**, since snow has a very distinct spectral signature (high reflectance across visible bands, low in SWIR)


> 💡 *Hint: if the 2018 vs 2019 separation disappears after adding noisy images, try normalizing or removing the outliers and re-running — what changes?*